# KeplerEllipse Trajectory (Starter)

This notebook shows the intended trajectory workflow using `PyAstronomy.pyasl.KeplerEllipse` as an external curve generator.

The pattern is:

1. Generate a trajectory curve (`X/Y/Z`, `V_x/V_y/V_z`, `t`).
2. Resample a source `SmartDs` onto the trajectory points.
3. Treat the result as a 1D curve `SmartDs` and continue from there.

This notebook assumes `PyAstronomy` is installed in the active environment.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.constants
from PyAstronomy import pyasl

from starwinds_analysis.smart_ds import SmartDs
from starwinds_analysis.data.samples import get_sample


## Load a 3D Source Dataset

This uses the 3D sample file and prepares the graph-backed field access.

In [ ]:
data_file = get_sample('3d__var_1_n00060000.plt')
sds = SmartDs.from_file(data_file)
sds.prepare()
print(sds)
star_radius_m = sds("star_radius [m]")
star_mass_kg = sds("star_mass [kg]")


## Build a Kepler Ellipse

The trajectory generator is external. Here we use `KeplerEllipse` and then convert the positions into stellar-radius units for resampling.

In [ ]:
semi_major_axis_r = 15.0
semi_major_axis_m = semi_major_axis_r * star_radius_m
eccentricity = 0.25
inclination_deg = 15.0

period_s = 2.0 * np.pi * np.sqrt(semi_major_axis_m**3 / (scipy.constants.G * star_mass_kg))
ke = pyasl.KeplerEllipse(
    a=semi_major_axis_m,
    per=period_s,
    e=eccentricity,
    i=inclination_deg,
)

n_points = 256
t_s = np.linspace(0.0, period_s, n_points, endpoint=False)
xyz_m = ke.xyzPos(t_s)
v_xyz_m_s = np.gradient(xyz_m, t_s, axis=0, edge_order=2)
xyz_r = xyz_m / star_radius_m


## Resample Onto the Trajectory Curve

This is the key step: the source `SmartDs` is sampled onto the trajectory points, and the trajectory-owned fields are appended to the returned curve dataset.

In [ ]:
curve = sds.resample(
    xyz_r,
    coordinate_fields=("X [R]", "Y [R]", "Z [R]"),
    fields=(
        "Rho [kg/m^3]",
        "U_x [m/s]",
        "U_y [m/s]",
        "U_z [m/s]",
    ),
    method="nearest",
    title="Kepler trajectory samples",
    zone="trajectory",
)

curve = curve.append_fields(
    {
        "V_x [m/s]": v_xyz_m_s[:, 0],
        "V_y [m/s]": v_xyz_m_s[:, 1],
        "V_z [m/s]": v_xyz_m_s[:, 2],
        "t [s]": t_s,
        "R_orbit [R]": np.sqrt(np.sum(xyz_r * xyz_r, axis=1)),
    },
    zone_suffix="trajectory",
)

print(curve)


## Use the Curve `SmartDs`

Once sampled, the trajectory is just another `SmartDs`, but with 1D point order.

Below, we use graph-backed `U_xyz` from the source data and the externally generated `V_xyz` to compute the relative speed along the trajectory.

In [ ]:
u_xyz_m_s = curve("U_xyz [m/s]")
v_xyz_m_s = np.column_stack((curve("V_x [m/s]"), curve("V_y [m/s]"), curve("V_z [m/s]")))
relative_speed_m_s = np.linalg.norm(u_xyz_m_s - v_xyz_m_s, axis=1)
rho = curve("Rho [kg/m^3]")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.0))
ax.plot(xyz_r[:, 0], xyz_r[:, 1], color="tab:blue")
ax.set_aspect("equal")
ax.set_xlabel("X [R]")
ax.set_ylabel("Y [R]")
ax.set_title("KeplerEllipse trajectory")
plt.show()


In [ ]:
fig, ax1 = plt.subplots(figsize=(8.0, 4.5))
time_days = curve("t [s]") / 86400.0
ax1.plot(time_days, rho, color="tab:green", label="Rho [kg/m^3]")
ax1.set_xlabel("t [days]")
ax1.set_ylabel("Rho [kg/m^3]")

ax2 = ax1.twinx()
ax2.plot(time_days, relative_speed_m_s / 1000.0, color="tab:red", label="|U - V| [km/s]")
ax2.set_ylabel("|U - V| [km/s]")

ax1.set_title("Sampled density and relative speed along the trajectory")
plt.show()
